# JetBot Road Following - Gamepad-Driven Data Collection

This is the road-following counterpart to the collision-avoidance data collection notebook: instead of labeling images `free` / `blocked` (classification), you'll place a target point (x, y) on each frame that says "steer toward here" (regression). In the training notebook, a network learns to predict that point; at deployment time the robot continuously chases the predicted point like a carrot on a stick.

Labeling guide:

1. Look at the live video feed from the camera.
2. Imagine the path the robot should follow.
3. Place the target as far along that path as it can go while still keeping the robot safely on the road/track if it drove straight at the target.

> On a straight stretch you can place the target near the horizon. On a sharp turn, place it closer to the robot so it doesn't run out of road.

This notebook adds gamepad automation on top of the stock `road_following/data_collection_gamepad.ipynb` pattern, and folds in the same drive controls from the collision-avoidance / teleop notebooks so you can drive and label without switching tools:

| Control | Action |
|---|---|
| D-pad Up | Drive forward at current speed |
| D-pad Down | Drive backward at current speed |
| D-pad Left | Pivot turn left |
| D-pad Right | Pivot turn right |
| Right stick X | Nudge target dot left / right |
| Right stick Y | Nudge target dot up / down |
| Right Bumper (RB) | Increase speed |
| Left Bumper (LB) | Decrease speed |
| **A button** | **Save current frame + (x, y) target to `dataset_xy`** |
| X button | Emergency stop + reset speed |

> **Note on button mapping:** the stock road-following notebook saves on D-pad Down (button 13). Since D-pad Down is needed here to drive backward, saving has been moved to the **A button** instead — matching the "A = capture" convention from the collision-avoidance version of this notebook. The left stick is unused in this notebook so both hands stay free for drive (D-pad, left thumb) and target placement (right stick, right thumb) at the same time.

> **Note on stick behavior:** the stock notebook links the stick directly to the dot's absolute position, so the dot snaps back to center the instant you let go of the stick. This version instead treats the right stick like a **cursor nudge**: tilting it moves the dot in that direction at a steady rate, and the dot **stays put** wherever you left it when the stick is released. This makes it much easier to hold a precise target position while driving.


### Labeling example video

This is the same reference clip from the stock road-following notebook, showing how the target point should be placed as you drive. Requires internet access on the Jetson to load; skip this cell if you're offline.

In [ ]:
from IPython.display import HTML
HTML('<iframe width="560" height="315" src="https://www.youtube.com/embed/FW4En6LejhI" frameborder="0" allow="accelerometer; autoplay; encrypted-media; gyroscope; picture-in-picture" allowfullscreen></iframe>')


## 1. Import libraries

In [ ]:
# IPython libraries for display and widgets
import traitlets
import ipywidgets.widgets as widgets
from IPython.display import display

# Camera and motor interface for JetBot
from jetbot import Robot, Camera, bgr8_to_jpeg

# Basic packages for image annotation
from uuid import uuid1
import os
import glob
import datetime
import numpy as np
import cv2
import time
import threading


## 2. Initialize the robot, camera, and target-dot display

Our neural network takes a 224x224 pixel image as input, so the camera is set to that size. Two linked image widgets are shown side by side: the raw feed, and a copy with the current target dot (green), robot origin (red), and steering line (blue) drawn on top so you can see exactly what will be saved.

In [ ]:
robot = Robot()
camera = Camera.instance(width=224, height=224)

image_widget = widgets.Image(format='jpeg', width=224, height=224)
target_widget = widgets.Image(format='jpeg', width=224, height=224)

x_slider = widgets.FloatSlider(min=-1.0, max=1.0, step=0.001, description='x')
y_slider = widgets.FloatSlider(min=-1.0, max=1.0, step=0.001, description='y')

def display_xy(camera_image):
    image = np.copy(camera_image)
    x = x_slider.value
    y = y_slider.value
    x = int(x * 224 / 2 + 112)
    y = int(y * 224 / 2 + 112)
    image = cv2.circle(image, (x, y), 8, (0, 255, 0), 3)
    image = cv2.circle(image, (112, 224), 8, (0, 0, 255), 3)
    image = cv2.line(image, (x, y), (112, 224), (255, 0, 0), 3)
    jpeg_image = bgr8_to_jpeg(image)
    return jpeg_image

camera_link = traitlets.dlink((camera, 'value'), (image_widget, 'value'), transform=bgr8_to_jpeg)
target_link = traitlets.dlink((camera, 'value'), (target_widget, 'value'), transform=display_xy)


## 3. Initialize the controller

Go to the gamepad tester website http://hardwaretester.com/gamepad and note the index number; if needed, edit the “ … Controller(index= )“ line in the cell.

In [ ]:
controller = widgets.Controller(index=0)  # change index if you have more than one gamepad connected
display(controller)


Before continuing: press a couple of buttons on the physical controller. The widget above needs at least one input event before `controller.buttons` / `controller.axes` populate. If it stays blank, make sure the browser has recognized the gamepad.

## 4. (Optional) Find your button and axis indices

Button and axis numbering isn't identical across every controller. Run this cell, then press buttons and move both joysticks one at a time, watching the output to confirm which index is which (D-pad, bumpers, A/X buttons, right stick X, right stick Y). Skip this step if you already know your controller's mapping.

In [ ]:
button_output = widgets.Output()
display(button_output)

def on_button_change(change, index):
    if change['new'] > 0:
        with button_output:
            print(f"Button {index} pressed (value={change['new']:.2f})")

def on_axis_change(change, index):
    with button_output:
        print(f"Axis {index} value={change['new']:.2f}")

for i, button in enumerate(controller.buttons):
    button.observe(lambda change, index=i: on_button_change(change, index), names='value')

for i, axis in enumerate(controller.axes):
    axis.observe(lambda change, index=i: on_axis_change(change, index), names='value')

print(f"Watching {len(controller.buttons)} buttons and {len(controller.axes)} axes.")
print("Press each button and move each stick fully up/down/left/right to identify indices.")


In [ ]:
# Run this once you're done identifying indices, so the diagnostic
# handlers above don't stick around and clutter behavior later.
for button in controller.buttons:
    button.unobserve_all()
for axis in controller.axes:
    axis.unobserve_all()


## 5. Configure the mapping and speeds

Edit the indices below to match what you found in step 4. Values shown are typical for an Xbox-style gamepad in Chrome, and match the indices used in the stock road-following gamepad notebook (axes 2/3 for the right stick).

In [ ]:
# --- Drive button mapping: adjust to match YOUR controller ---
BTN_FORWARD  = 12   # D-pad Up
BTN_BACKWARD = 13   # D-pad Down
BTN_LEFT     = 14   # D-pad Left
BTN_RIGHT    = 15   # D-pad Right
BTN_SPEED_UP = 5    # Right bumper (RB)
BTN_SPEED_DN = 4    # Left bumper (LB)
BTN_ESTOP    = 2    # X button

# --- Data collection button mapping ---
BTN_SAVE     = 0    # A button -> save current frame + (x, y) target

# --- Axis mapping for the target dot: adjust to match YOUR controller ---
AXIS_TARGET_X = 2   # Right stick, horizontal
AXIS_TARGET_Y = 3   # Right stick, vertical

# --- Target dot nudge settings ---
# The dot moves like a cursor: holding the stick over nudges it steadily in that
# direction, and it stays wherever you left it when the stick is released (it does
# NOT snap back to center like a direct absolute-position mapping would).
NUDGE_DEADZONE  = 0.15  # ignore stick movement smaller than this (drift/noise)
NUDGE_RATE      = 0.5   # slider units/second at full stick deflection (slider range is -1..1)
NUDGE_INTERVAL  = 0.05  # seconds between nudge updates (20 Hz)

# --- Speed settings ---
DEFAULT_SPEED = 0.2
DEFAULT_TURN_SPEED = 0.35
SPEED_STEP = 0.1
SPEED_MIN = 0.1
SPEED_MAX = 1.0

speed = DEFAULT_SPEED
turn_speed = DEFAULT_TURN_SPEED


## 6. Set up the dataset folder and image counter

Images are saved as ``xy_<x>_<y>_<uuid>.jpg`` in `dataset_xy`, matching the stock road-following notebook so the existing `train_model.ipynb` can parse the x, y values straight out of the filename.

In [ ]:
DATASET_DIR = 'dataset_xy'

# we have this "try/except" statement because this can throw an error if the directory exists already
try:
    os.makedirs(DATASET_DIR)
except FileExistsError:
    print('Directory not created because it already exists')

count_widget = widgets.IntText(description='count', value=len(glob.glob(os.path.join(DATASET_DIR, '*.jpg'))))


## 7. Wire the D-pad, right stick, and save button

In [ ]:
DIRECTION_BUTTONS = [BTN_FORWARD, BTN_BACKWARD, BTN_LEFT, BTN_RIGHT]

def update_motion(change=None):
    """Reads the current state of the four direction buttons and drives
    the robot accordingly. Called any time one of them is pressed or released."""
    up    = controller.buttons[BTN_FORWARD].value  > 0.5
    down  = controller.buttons[BTN_BACKWARD].value > 0.5
    left  = controller.buttons[BTN_LEFT].value     > 0.5
    right = controller.buttons[BTN_RIGHT].value    > 0.5

    if up and not down:
        robot.forward(speed)
    elif down and not up:
        robot.backward(speed)
    elif left and not right:
        robot.left(turn_speed)
    elif right and not left:
        robot.right(turn_speed)
    else:
        robot.stop()

def change_speed(change, delta):
    global speed, turn_speed
    if change['new'] > 0.5:  # only act on press, not release
        speed = round(max(SPEED_MIN, min(SPEED_MAX, speed + delta)), 2)
        turn_speed = round(max(SPEED_MIN, min(SPEED_MAX, turn_speed + delta)), 2)
        print(f"Speed: {speed:.2f}   Turn speed: {turn_speed:.2f}")
        update_motion()  # re-apply D-pad state if a direction is held

def emergency_stop(change):
    global speed, turn_speed
    if change['new'] > 0.5:
        robot.stop()
        speed = DEFAULT_SPEED
        turn_speed = DEFAULT_TURN_SPEED
        print("EMERGENCY STOP - speed reset to defaults")

def nudge_loop():
    """Runs in a background thread at ~NUDGE_INTERVAL. Reads the current stick
    deflection and nudges the target dot in that direction; does nothing when the
    stick is centered, so the dot simply holds its last position on release."""
    while nudge_state['running']:
        ax = controller.axes[AXIS_TARGET_X].value
        ay = controller.axes[AXIS_TARGET_Y].value
        step = NUDGE_RATE * NUDGE_INTERVAL
        if abs(ax) > NUDGE_DEADZONE:
            x_slider.value = max(x_slider.min, min(x_slider.max, x_slider.value + ax * step))
        if abs(ay) > NUDGE_DEADZONE:
            y_slider.value = max(y_slider.min, min(y_slider.max, y_slider.value + ay * step))
        time.sleep(NUDGE_INTERVAL)

def xy_uuid(x, y):
    return 'xy_%03d_%03d_%s' % (x * 50 + 50, y * 50 + 50, uuid1())

def save_snapshot():
    uuid = xy_uuid(x_slider.value, y_slider.value)
    image_path = os.path.join(DATASET_DIR, uuid + '.jpg')
    with open(image_path, 'wb') as f:
        f.write(image_widget.value)
    count_widget.value = len(glob.glob(os.path.join(DATASET_DIR, '*.jpg')))
    print(f"Saved ({count_widget.value} total) x={x_slider.value:.3f} y={y_slider.value:.3f}")

def on_save_button(change):
    if change['new'] > 0.5:  # only act on press, not release
        save_snapshot()

# Direction buttons
for i in DIRECTION_BUTTONS:
    controller.buttons[i].observe(update_motion, names='value')

# Right stick -> target dot nudging (background thread, see nudge_loop above)
nudge_state = {'running': True}
nudge_thread = threading.Thread(target=nudge_loop, daemon=True)
nudge_thread.start()

# Speed buttons
controller.buttons[BTN_SPEED_UP].observe(lambda c: change_speed(c, SPEED_STEP), names='value')
controller.buttons[BTN_SPEED_DN].observe(lambda c: change_speed(c, -SPEED_STEP), names='value')

# Emergency stop
controller.buttons[BTN_ESTOP].observe(emergency_stop, names='value')

# Save button
controller.buttons[BTN_SAVE].observe(on_save_button, names='value')

print("Controller wired up:")
print("  D-pad          -> drive / turn (both wheels together)")
print("  Right stick     -> nudge target dot (x, y), holds position on release")
print("  RB / LB        -> speed up / down")
print("  X              -> emergency stop")
print("  A              -> save frame + target (x, y)")
print(f"Starting speed: {speed:.2f}   Starting turn speed: {turn_speed:.2f}")


## 8. Drive around and collect data

Drive with the D-pad, use the right stick to slide the green target dot to where the robot should steer, and tap **A** the moment it's placed correctly. The right-hand image below shows exactly what gets saved: green dot = target, red dot = robot origin, blue line = the implied steering line.

In [ ]:
display(widgets.HBox([image_widget, target_widget]))
display(widgets.HBox([x_slider, y_slider]))
display(count_widget)


## 9. Disconnect when finished

Run this before closing the notebook or restarting the kernel, so the robot doesn't keep responding to a controller that's no longer being watched, and so the camera is released for the next notebook (e.g. `train_model.ipynb` or a live demo notebook).

In [ ]:
for i in DIRECTION_BUTTONS + [BTN_SPEED_UP, BTN_SPEED_DN, BTN_ESTOP, BTN_SAVE]:
    controller.buttons[i].unobserve_all()
nudge_state['running'] = False   # stops the nudge_loop background thread
robot.stop()
camera.stop()
print("Disconnected. Robot stopped, camera released.")


## 10. Package the dataset

Once you've collected enough data, zip it up for transfer to the GPU machine running `road_following/train_model.ipynb`. Skip this step if you're training on the JetBot itself.

In [ ]:
def timestr():
    return str(datetime.datetime.now().strftime('%Y-%m-%d_%H-%M-%S'))

!zip -r -q road_following_{DATASET_DIR}_{timestr()}.zip {DATASET_DIR}
